In [ ]:
input_data = None
targetpop_data = None
output_data = None
output_model = None
util = None
display_util = None
configfile = "config/config.yml"

In [ ]:
import yaml

with open(configfile) as stream:
    config = yaml.safe_load(stream)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline
import matplotlib.pyplot as plt

from IPython.display import Markdown
import pandera.pandas as pa
from pandera.typing import Series

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
plt.ioff()
plt.rcParams["figure.figsize"] = (8, 5)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rule_setup,
    display_data_doc,
    display_long_data_doc,
    collist,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    SpenderID,
    drop_duplicate_columns,
    common_translate,
    split_data,
    collapse_col,
    find_redundant_cols,
    fix_redundancies,
)

### Target Population Filtering

The donors in the dataset were filtered to match the target population (see [](general:tpf)). Afterwards we tried again to remove empty and duplicate columns.

In [ ]:
data = pd.read_parquet(input_data)
donors = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
targetpop = pd.read_parquet(targetpop_data)
data = data[donors.isin(targetpop["donor_et_id_et"])]
display(
    Markdown(
        f"""The filter process reduced the number of donors in the data ({donors.nunique()}) and target population ({targetpop["donor_et_id_et"].nunique()})
            to {donors[donors.isin(targetpop["donor_et_id_et"])].nunique()} in the processed data.
        """
    )
)

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data)

### Integration of Seperated Institute Data

In this file, the {term}`DSO` and {term}`ET` data is not connected (see [](general:ic)). No joining process is necessary, as the data is in the longitudinal format.

In [ ]:
idcols = ["donor_et_dso", "donor_et_id_et"]
assert len(split_data(data, idcols)) == 3, "Not 3 different row types present!?"
assert (
    data.loc[:, ["sampling_date_et", "sampling_date_dso"]]
    .diff(axis=1)
    .iloc[:, 1]
    .dropna()
    == 0
).all(), "Dates sometimes different"

## Domain Steps

For this file the general plan for domain preprocessing of longitudinal data was followed (see [](general:ds)).

### Row Filtering

There is no column differentiating between different types of tests (see [](general:rf)). The following analysis compares the data from the different sources. All rows were kept.

In [ ]:
data["Institute with a measurement date"] = (
    (~data["sampling_date_dso"].isna()) + (~data["sampling_date_et"].isna()) * 2
).replace({1: "DSO", 2: "ET", 3: "DSO+ET", 0: "No Date"})
data["donor"] = collapse_col(
    data.loc[:, ["donor_et_dso", "donor_et_id_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
data["sampling_date"] = collapse_col(
    data.loc[:, ["sampling_date_dso", "sampling_date_et"]],
    fun=lambda row: None if row.nunique() > 1 else row.iloc[0],
)
display_long_data_doc(
    data,
    [
        "donor",
    ],
    "sampling_date",
    "Institute with a measurement date",
)
data.drop(
    columns=["sampling_date", "donor", "Institute with a measurement date"],
    inplace=True,
)

### Unit Conversions

First common translations were applied and afterwards, unit specifier columns with only a single unit were removed (see [](general:uc)).

In [ ]:
data = common_translate(data, config["data"]["common_translations"])

In [ ]:
cols = data.columns[data.columns.to_series().str.contains("_unit")]
assert (data[cols].nunique() != 1).sum() == 0
dropme = cols[data[cols].nunique() == 1]
data = data.drop(columns=dropme)
display(
    Markdown(
        f"The columns {collist(dropme)} were removed as only a single unit was used."
    )
)

### Consolidating Columns

We consolidated columns that appear for {term}`ET` and {term}`DSO` (see [](general:crc))

In [ ]:
red = find_redundant_cols(data)
red["donor_et_id_et"] = ["donor_et_dso", "donor_et_id_et"]
fix_redundancies(data, red)

## Intermediate Dataset

For this longitudinal dataset we recommend the `sampling_date` column as the time axis.

In [ ]:
indcols = ["donor_et_id_et"]
data = data.sort_index(axis=1).sort_values(indcols + ["sampling_date"], axis=0)
data = data.set_index(indcols)

In [ ]:
labtestvals = ["not tested", "positive", "negative"]


class DonorPostmortemLabVirology(SpenderID):
    communicated_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Communicated date",
        description="Date when the lab result was communicated",
    )
    cytomegalovirus_igg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="CMV IgG Antibodies Detected",
        description="Were cytomegalovirus IgG antibodies present?",
        isin=labtestvals,
    )
    cytomegalovirus_igm: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="CMV IgM Antibodies Detected",
        description="Were cytomegalovirus IgG antibodies present?",
        isin=labtestvals,
    )
    epstein_barr_virus_igg: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="EBV IgG Antibodies Detected",
        description="Were Epstein Barr virus IgG antibodies present?",
        isin=labtestvals,
    )
    examination_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Examination date",
        description="Date when the lab measurements were conducted",
    )
    hepatitis_b_core_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B core antibodies detected",
        description="Were Hepatitis B core antibodies present?",
        isin=labtestvals,
    )
    hepatitis_b_core_igm: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B core IgM Antibodies Detected",
        description="Were Hepatitis B core IgM antibodies present?",
        isin=labtestvals,
    )
    hepatitis_b_dna: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B DNA",
        description="Was Hepatitis B DNA present?",
        isin=labtestvals,
    )
    hepatitis_b_surface_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B surface antibodies detected",
        description="Were Hepatitis B surface antibodies present?",
        isin=labtestvals,
    )
    hepatitis_b_surface_antibodies_iu_per_l: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B surface antibodies concentration",
        description="Concentration Hepatitis B surface antibodies present in IU/l",
    )
    hepatitis_b_surface_antigens: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis B surface antigens detected",
        description="Were Hepatitis B surface antigens present?",
        isin=labtestvals,
    )
    hepatitis_c_akr: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis C akr (?) detected",
        description="Were Hepatitis C akr (?) present?",
        isin=labtestvals,
    )
    hepatitis_c_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis C antibodies detected",
        description="Were Hepatitis C surface antibodies present?",
        isin=labtestvals,
    )
    hepatitis_c_rna: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Hepatitis C RNA detected",
        description="Was Hepatitis C RNA resent?",
        isin=labtestvals,
    )
    hiv_antibodies: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV antibodies detected",
        description="Were surface antibodies present?",
        isin=labtestvals,
    )
    hiv_antibody_westernblot: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV antibodies detected with Western Blot",
        description="Were surface antibodies present with Western Blot?",
        isin=labtestvals,
    )
    hiv_antigens: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV antigens detected",
        description="Were HIV antigens present?",
        isin=labtestvals,
    )
    hiv_rna1: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV 1 RNA detected",
        description="Was HIV 1 RNA present?",
        isin=labtestvals,
    )
    hiv_rna2: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="HIV 2 RNA detected",
        description="Was HIV 2 RNA present?",
        isin=labtestvals,
    )
    meningitis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Mengitis",
        description="Was a mengitis diagnosedt?",
        isin=["no", "yes", "unknown"],
    )
    result_date: Series[float] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Result date",
        description="Date when the lab result was generated",
    )
    sampling_date: Series[float] = pa.Field(
        coerce=True,
        nullable=False,
        unique=False,
        title="Sampling date",
        description="Date when the sample was taken",
    )
    sampling_tissue: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Tissue",
        description="From what tissue was the sample taken?",
        isin=[
            "Blut",
            "Serum",
            "Abstrich",
            "Heparinblut",
            "Biopsie",
            "Bronchiallavage",
            "Trachealsekret",
            "Liquor",
        ],
    )
    sepsis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Sepsis",
        description="Was a sepsis diagnosed?",
        isin=["no", "yes", "unknown"],
    )
    syphilis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Syphilis diagnosed",
        description="Was syphilis diagnosed?",
        isin=labtestvals,
    )
    toxoplasmosis: Series[str] = pa.Field(
        coerce=True,
        nullable=True,
        unique=False,
        title="Toxoplasmosis diagnosed",
        description="Was toxoplasmosis diagnosed?",
        isin=labtestvals,
    )

    class Config:
        title = "Donor Postmortem Urine Virology Test Dataset"
        description = "Each row represents a virus lab test for a deceased donor. The data is based on the 'element_spender_postmortem_labor_virologie.csv' file. It contains data from the DSO and ET."
        multiindex_strict = True
        multiindex_coerce = True

In [ ]:
display_data_doc(DonorPostmortemLabVirology, data)

In [ ]:
DonorPostmortemLabVirology.to_schema().validate(data).to_parquet(output_data)
with open(output_model, "wt") as fh:
    DonorPostmortemLabVirology.to_yaml(stream=fh)

## Technical Information

In [ ]:
rule_setup(
    {
        "input_data": input_data,
        "targetpop_data": targetpop_data,
        "output_data": output_data,
        "output_model": output_model,
        "util": util,
        "display_util": display_util,
    }
)